In [0]:
import pyspark.sql.types as T
import pyspark.sql.functions as F

In [0]:
catalog_name = 'ecommerce'

In [0]:
df = spark.table(f"{catalog_name}.silver.slv_order_items")

df.limit(10).display()

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,file_name,ingest_timestamp,processed_time
2025-08-01,2025-08-01T22:53:52.000Z,CUST000000241190,643611,1,2000000028279,1,GBP,11.0,10.0,2.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z
2025-08-01,2025-08-01T22:53:52.000Z,CUST000000241190,643611,2,2000000377445,1,GBP,5.0,4.0,1.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z
2025-08-01,2025-08-01T09:44:32.000Z,CUST000000239553,643612,1,2000000417639,4,INR,4871.0,8.0,3245.0,Website,fest20,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,1,2000000422664,2,AED,74.0,0.0,18.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,2,2000000238159,1,AED,1012.0,11.0,109.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,3,2000000094205,2,AED,229.0,15.0,20.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,4,2000000293790,2,AED,2222.0,0.0,801.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z
2025-08-01,2025-08-01T21:09:10.000Z,CUST000000117094,643614,1,2000000271415,3,INR,252.0,10.0,82.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z
2025-08-01,2025-08-01T13:03:57.000Z,CUST000000016826,643615,1,2000000050980,1,GBP,71.0,5.0,9.0,Website,save50,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z
2025-08-01,2025-08-01T17:54:51.000Z,CUST000000252796,643616,1,2000000463254,1,USD,6.0,9.0,1.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z


### Gross amount


In [0]:
df = df.withColumn(
    "gross_amount",
    F.col("quantity") * F.col("unit_price")
    )

### Discount amount

In [0]:
df = df.withColumn(
    "discount_amount",
    F.ceil(F.col("gross_amount") * (F.col("discount_pct") / 100.0))
)

### Sale amount

In [0]:
df = df.withColumn(
    "sale_amount",
    F.col("gross_amount") - F.col("discount_amount") + F.col("tax_amount")
)

In [0]:
df = df.withColumn("date_id", F.date_format(F.col("dt"), "yyyyMMdd").cast(T.IntegerType())) 

### Coupon Flag column -- coupon flag = 1 if not null else 0

In [0]:
df = df.withColumn(
    "coupon_flag",
    F.when(F.col("coupon_code").isNotNull(), F.lit(1))
     .otherwise(F.lit(0))
)

In [0]:
df.limit(5).display()  

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,file_name,ingest_timestamp,processed_time,gross_amount,discount_amount,sale_amount,date_id,coupon_flag
2025-08-01,2025-08-01T22:53:52.000Z,CUST000000241190,643611,1,2000000028279,1,GBP,11.0,10.0,2.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z,11.0,2,11.0,20250801,0
2025-08-01,2025-08-01T22:53:52.000Z,CUST000000241190,643611,2,2000000377445,1,GBP,5.0,4.0,1.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z,5.0,1,5.0,20250801,0
2025-08-01,2025-08-01T09:44:32.000Z,CUST000000239553,643612,1,2000000417639,4,INR,4871.0,8.0,3245.0,Website,fest20,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z,19484.0,1559,21170.0,20250801,1
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,1,2000000422664,2,AED,74.0,0.0,18.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z,148.0,0,166.0,20250801,0
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,2,2000000238159,1,AED,1012.0,11.0,109.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z,1012.0,112,1009.0,20250801,0


In [0]:
fx_rates = {
    "INR": 1.00,
    "AED": 24.18,
    "AUD": 57.55,
    "CAD": 62.93,
    "GBP": 117.98,
    "SGD": 68.18,
    "USD": 88.29,
}

rates = [(k, float(v)) for k, v in fx_rates.items()]
rates_df = spark.createDataFrame(rates, ["currency", "inr_rate"])
rates_df.show()

+--------+--------+
|currency|inr_rate|
+--------+--------+
|     INR|     1.0|
|     AED|   24.18|
|     AUD|   57.55|
|     CAD|   62.93|
|     GBP|  117.98|
|     SGD|   68.18|
|     USD|   88.29|
+--------+--------+



In [0]:
df = (
    df
    .join(
        rates_df,
        rates_df.currency == F.upper(F.trim(F.col("unit_price_currency"))),
        "left"
    )
    .withColumn("sale_amount_inr", F.col("sale_amount") * F.col("inr_rate"))
    .withColumn("sale_amount_inr", F.ceil(F.col("sale_amount_inr")))
)

In [0]:
df.limit(5).display()

dt,order_ts,customer_id,order_id,item_seq,product_id,quantity,unit_price_currency,unit_price,discount_pct,tax_amount,channel,coupon_code,file_name,ingest_timestamp,processed_time,gross_amount,discount_amount,sale_amount,date_id,coupon_flag,currency,inr_rate,sale_amount_inr
2025-08-01,2025-08-01T22:53:52.000Z,CUST000000241190,643611,1,2000000028279,1,GBP,11.0,10.0,2.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z,11.0,2,11.0,20250801,0,GBP,117.98,1298
2025-08-01,2025-08-01T22:53:52.000Z,CUST000000241190,643611,2,2000000377445,1,GBP,5.0,4.0,1.0,Website,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z,5.0,1,5.0,20250801,0,GBP,117.98,590
2025-08-01,2025-08-01T09:44:32.000Z,CUST000000239553,643612,1,2000000417639,4,INR,4871.0,8.0,3245.0,Website,fest20,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z,19484.0,1559,21170.0,20250801,1,INR,1.0,21170
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,1,2000000422664,2,AED,74.0,0.0,18.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z,148.0,0,166.0,20250801,0,AED,24.18,4014
2025-08-01,2025-08-01T04:45:03.000Z,CUST000000175269,643613,2,2000000238159,1,AED,1012.0,11.0,109.0,Mobile,null,dbfs:/Volumes/ecommerce/source_data/raw/order_items/landing/order_items_2025-08-01.csv,2026-09-20T19:46:14.363Z,2026-09-20T20:24:14.346Z,1012.0,112,1009.0,20250801,0,AED,24.18,24398


In [0]:
orders_gold_df = df.select(
    F.col("date_id"),
    F.col("dt").alias("transaction_date"),
    F.col("order_ts").alias("transaction_ts"),
    F.col("order_id").alias("transaction_id"),
    F.col("customer_id"),
    F.col("item_seq").alias("seq_no"),
    F.col("product_id"),
    F.col("channel"),
    F.col("coupon_code"),
    F.col("coupon_flag"),
    F.col("unit_price_currency"),
    F.col("quantity"),
    F.col("unit_price"),
    F.col("gross_amount"),
    F.col("discount_pct").alias("discount_percent"),
    F.col("discount_amount"),
    F.col("tax_amount"),
    F.col("sale_amount").alias("net_amount"),
    F.col("sale_amount_inr").alias("net_amount_inr")
)

In [0]:
orders_gold_df.limit(5).display()

date_id,transaction_date,transaction_ts,transaction_id,customer_id,seq_no,product_id,channel,coupon_code,coupon_flag,unit_price_currency,quantity,unit_price,gross_amount,discount_percent,discount_amount,tax_amount,net_amount,net_amount_inr
20250801,2025-08-01,2025-08-01T22:53:52.000Z,643611,CUST000000241190,1,2000000028279,Website,null,0,GBP,1,11.0,11.0,10.0,2,2.0,11.0,1298
20250801,2025-08-01,2025-08-01T22:53:52.000Z,643611,CUST000000241190,2,2000000377445,Website,null,0,GBP,1,5.0,5.0,4.0,1,1.0,5.0,590
20250801,2025-08-01,2025-08-01T09:44:32.000Z,643612,CUST000000239553,1,2000000417639,Website,fest20,1,INR,4,4871.0,19484.0,8.0,1559,3245.0,21170.0,21170
20250801,2025-08-01,2025-08-01T04:45:03.000Z,643613,CUST000000175269,1,2000000422664,Mobile,null,0,AED,2,74.0,148.0,0.0,0,18.0,166.0,4014
20250801,2025-08-01,2025-08-01T04:45:03.000Z,643613,CUST000000175269,2,2000000238159,Mobile,null,0,AED,1,1012.0,1012.0,11.0,112,109.0,1009.0,24398


In [0]:
orders_gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_fact_order_items")